# 🧠 NLP Foundations Workshop: Vector Space Proximity
## PROG 8245 — Machine Learning Programming | Lab 7

**Team:** Group 6  
**Members:** Emmanuel Ihejiamaizu · Liggia Elena Taboada Cruz · Chao-Chung Liu (Thomas)  
**Program:** Graduate Diploma in Applied AI & Machine Learning — Conestoga College  
**Workshop:** IR Basics & Vector Space Proximity  

---

**Corpus:** CISA Known Exploited Vulnerabilities (KEV) Catalog  
**Source:** https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv  
**License:** U.S. Government Open Data — Public Domain  
**Domain:** Cybersecurity — Real CVE vulnerability records published by CISA  


## Environment Setup

```
python -m venv venv
venv\Scripts\activate
pip install numpy pandas scikit-learn matplotlib seaborn nltk jupyter
```

Place `known_exploited_vulnerabilities.csv` in the same folder as this notebook.


## Consolidated Imports

In [ ]:
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA
from sklearn.metrics import (confusion_matrix, f1_score,
                              precision_score, recall_score,
                              accuracy_score, cohen_kappa_score)
from nltk.stem import PorterStemmer

print("All libraries loaded successfully.")


## Part A — Corpus Description

### Why We Selected This Dataset

The **CISA Known Exploited Vulnerabilities (KEV) Catalog** is published by the U.S. Cybersecurity and Infrastructure Security Agency — the authoritative federal body for civilian cybersecurity. It lists every vulnerability that has been actively exploited in the wild.

We selected this dataset because:
- It is a **real, government-verified** collection — not a synthetic corpus
- Every document is a **concise technical description** of a distinct vulnerability
- The `knownRansomwareCampaignUse` column provides **built-in relevance labels** derived from threat intelligence
- It directly connects to cybersecurity — our team's specialisation

### Corpus Statistics

| Property | Value |
|---|---|
| Source | CISA KEV Catalog (U.S. Government) |
| Total CVE records | 1,551 |
| Documents used | 200 (random sample, seed=42) |
| Relevant documents | 40 (knownRansomwareCampaignUse == 'Known') |
| Not Relevant | 160 (knownRansomwareCampaignUse == 'Unknown') |
| Document type | Short vulnerability descriptions (~27 words avg) |
| Domain | Cybersecurity — CVE vulnerability records |


## Load Dataset — No Hardcoding

In [ ]:
# Load from CSV — corpus text, topics, and relevance labels all come from the file
df_full = pd.read_csv('known_exploited_vulnerabilities.csv')

# Sample 200 documents — random_state=42 ensures reproducibility
df = df_full.sample(200, random_state=42).reset_index(drop=True)

# Extract columns — zero hardcoded text
documents      = df['shortDescription'].tolist()
topics         = df['vulnerabilityName'].tolist()
cve_ids        = df['cveID'].tolist()
vendors        = df['vendorProject'].tolist()
relevant_flags = (df['knownRansomwareCampaignUse'] == 'Known').tolist()
truly_relevant = set(i for i, r in enumerate(relevant_flags) if r)
total_relevant = len(truly_relevant)

print(f"Total documents loaded  : {len(documents)}")
print(f"Relevant (Known)        : {total_relevant}")
print(f"Not Relevant (Unknown)  : {len(documents) - total_relevant}")
print(f"Vocabulary (raw approx) : {len(set(' '.join(documents).lower().split()))} unique words")
print()
print("Sample documents:")
for i in range(3):
    label = 'RELEVANT' if i in truly_relevant else 'not relevant'
    print(f"  [{cve_ids[i]}] ({label})")
    print(f"  Vendor: {vendors[i]}")
    print(f"  {documents[i][:110]}...")
    print()


## Part B — Preprocessing Pipeline

A preprocessing pipeline transforms raw text into a clean, normalised token sequence before any vectorisation step.

The pipeline has four stages:
1. **Tokenization** — split text into individual words
2. **Normalisation** — lowercase, remove punctuation and non-alpha characters
3. **Stop-word removal** — drop common words that carry no discriminative value (e.g. *the*, *is*, *a*)
4. **Stemming** — reduce words to their root form (e.g. *exploiting* → *exploit*, *vulnerabilities* → *vulner*)


In [ ]:
# --- Preprocessing Pipeline ---
stemmer    = PorterStemmer()
stop_words = set(ENGLISH_STOP_WORDS)   # sklearn's built-in English stop words

def tokenize(text: str):
    """Stage 1 — Split text into lowercase alphabetic tokens."""
    return re.findall(r'[a-z]+', text.lower())

def remove_stopwords(tokens: list):
    """Stage 2 — Remove stop words and very short tokens (< 3 chars)."""
    return [t for t in tokens if t not in stop_words and len(t) > 2]

def stem(tokens: list):
    """Stage 3 — Apply Porter stemming to reduce to root forms."""
    return [stemmer.stem(t) for t in tokens]

def preprocess(text: str):
    """Full pipeline: tokenize → remove stopwords → stem."""
    return stem(remove_stopwords(tokenize(text)))

# --- Demonstrate on first 3 documents ---
print("Preprocessing demonstration:")
print(f"{'Stage':<25} {'Example output'}")
print("-" * 75)
doc = documents[0]
print(f"{'Raw text':<25} {doc[:80]}...")
t1  = tokenize(doc)
print(f"{'After tokenize':<25} {t1[:10]}")
t2  = remove_stopwords(t1)
print(f"{'After stopword removal':<25} {t2[:10]}")
t3  = stem(t2)
print(f"{'After stemming':<25} {t3[:10]}")
print()

# --- Build preprocessed corpus ---
preprocessed_docs = [preprocess(doc) for doc in documents]
all_tokens        = [t for doc in preprocessed_docs for t in doc]
vocab             = sorted(set(all_tokens))

print(f"Preprocessed vocabulary size : {len(vocab)}")
print(f"Total tokens across corpus   : {len(all_tokens)}")
print(f"Avg tokens per document      : {len(all_tokens)/len(documents):.1f}")


### Interpretation
- **Tokenization** isolates individual words and discards numbers, punctuation, and casing differences.
- **Stop-word removal** eliminates words like *contains*, *allows*, *the* that appear in almost every CVE description and would add noise rather than signal.
- **Stemming** ensures that *execute*, *execution*, and *executing* are treated as the same token — critical for vulnerability text where the same concept appears in many grammatical forms.
- After preprocessing, the vocabulary shrinks significantly, making downstream vectorisation more focused on meaningful cybersecurity terms.


## Step 1 — Term-Document Incidence Matrix

The **Term-Document Incidence Matrix** is the simplest document representation:  
- Rows = documents  
- Columns = vocabulary terms  
- Cell value = **1** if the term appears in the document, **0** otherwise

This is a **binary** representation — it ignores how many times a term appears.


In [ ]:
# Show incidence matrix for first 6 documents, top 15 terms
cv_binary  = CountVectorizer(binary=True, stop_words='english', max_features=15)
incidence  = cv_binary.fit_transform(documents[:6]).toarray()
vocab_bin  = cv_binary.get_feature_names_out()

df_incidence = pd.DataFrame(
    incidence,
    index=[f"Doc {i} [{cve_ids[i][:12]}]" for i in range(6)],
    columns=vocab_bin
)

print("Term-Document Incidence Matrix (first 6 docs, 15 terms):")
print(df_incidence.to_string())
print()
print("Reading example:")
print(f"  Doc 0 contains 'execution': {incidence[0][list(vocab_bin).index('execution')] if 'execution' in vocab_bin else 'term not in top 15'}")
print(f"  Matrix entry = 1 (present) or 0 (absent) — no count, just presence")


### Interpretation
- Each row is a **binary vector** representing one document.
- Two documents with identical incidence vectors are perfectly similar in this representation, even if one mentions a term once and the other 20 times.
- This is the foundation of Boolean retrieval — query terms are matched against the incidence matrix using AND/OR/NOT logic.
- The limitation is clear: **all present terms are treated equally**, regardless of frequency or importance.


## Step 2 — Term Frequency (TF)

**Term Frequency** counts how many times a term appears in a document:

$$tf(t, d) = \text{count of term } t \text{ in document } d$$

Unlike the incidence matrix, TF captures how strongly a term dominates a document.


In [ ]:
# Build TF matrix for top 15 terms
cv_tf    = CountVectorizer(stop_words='english', max_features=15)
tf_mat   = cv_tf.fit_transform(documents).toarray().astype(float)
vocab_tf = cv_tf.get_feature_names_out()

# Show TF for first 6 documents
df_tf = pd.DataFrame(
    tf_mat[:6],
    index=[f"Doc {i} [{cve_ids[i][:12]}]" for i in range(6)],
    columns=vocab_tf
)

print("Term Frequency Matrix (first 6 docs, 15 terms):")
print(df_tf.to_string())
print()

# Show most frequent terms across full corpus
term_totals = tf_mat.sum(axis=0)
top_terms   = sorted(zip(vocab_tf, term_totals), key=lambda x: -x[1])[:10]
print("Most frequent terms across entire corpus:")
for term, freq in top_terms:
    print(f"  {term:<20} total TF = {int(freq)}")


### Interpretation
- Terms like *vulnerability*, *attacker*, and *remote* appear frequently because they are common to all CVE descriptions — high TF but low discriminative power.
- Terms like *ransomware*, *injection*, or *overflow* are rarer and more topic-specific — lower TF but higher information value.
- This motivates the IDF adjustment in the next steps.


## Step 3 — Log Frequency Weighting

Raw TF is not ideal — a term appearing 10 times is not necessarily 10× more important than one appearing once.

**Log TF** dampens the effect of high-frequency terms:

$$w(t, d) = \begin{cases} 1 + \log_{10}(tf(t,d)) & \text{if } tf(t,d) > 0 \\ 0 & \text{otherwise} \end{cases}$$


In [ ]:
# Compute log TF from the raw TF matrix
log_tf = np.where(tf_mat > 0, 1 + np.log(tf_mat + 1e-9), 0)

df_log_tf = pd.DataFrame(
    log_tf[:6],
    index=[f"Doc {i} [{cve_ids[i][:12]}]" for i in range(6)],
    columns=vocab_tf
)

print("Log TF Matrix (first 6 docs, 15 terms):")
print(df_log_tf.round(3).to_string())
print()

# Compare raw TF vs log TF for a sample term
sample_term = 'vulnerability'
if sample_term in list(vocab_tf):
    idx = list(vocab_tf).index(sample_term)
    print(f"Comparison for term '{sample_term}':")
    print(f"{'Doc':<6} {'Raw TF':>10} {'Log TF':>10}")
    print("-" * 30)
    for i in range(6):
        print(f"{i:<6} {tf_mat[i,idx]:>10.0f} {log_tf[i,idx]:>10.4f}")


### Interpretation
- When raw TF = 1, Log TF ≈ 1.0
- When raw TF = 5, Log TF ≈ 1.7 (not 5×)
- When raw TF = 10, Log TF ≈ 2.3 (not 10×)
- Log weighting prevents a single term dominating the vector just because it is repeated many times in a short vulnerability description.


## Step 4 — Document Frequency (DF)

**Document Frequency** counts how many documents contain a given term:

$$df_t = \text{number of documents containing term } t$$

Terms with very high DF appear in almost every document — they carry little discriminative power for retrieval.


In [ ]:
# Compute DF — how many documents each term appears in
df_counts = (tf_mat > 0).sum(axis=0)
N         = len(documents)

df_df = pd.DataFrame({
    'Term'             : vocab_tf,
    'DF'               : df_counts.astype(int),
    'DF%'              : (df_counts / N * 100).round(1)
}).sort_values('DF', ascending=False).reset_index(drop=True)

print(f"Document Frequency — top 15 terms (N = {N} documents):")
print(df_df.to_string(index=False))
print()
print("Interpretation:")
print(f"  '{df_df.iloc[0]['Term']}' appears in {df_df.iloc[0]['DF']} / {N} documents ({df_df.iloc[0]['DF%']}%)")
print(f"  These ubiquitous terms are poor discriminators — IDF will suppress them.")


## Step 5 — Inverse Document Frequency (IDF)

**IDF** penalises terms that appear in many documents and rewards rare, informative terms:

$$idf_t = \log\left(\frac{N+1}{df_t+1}\right) + 1$$

- Common terms (high DF) → low IDF → less weight
- Rare terms (low DF) → high IDF → more weight


In [ ]:
# Compute IDF
idf_vals = np.log((N + 1) / (df_counts + 1)) + 1

df_idf = pd.DataFrame({
    'Term' : vocab_tf,
    'DF'   : df_counts.astype(int),
    'IDF'  : idf_vals.round(4)
}).sort_values('IDF', ascending=False).reset_index(drop=True)

print("IDF values — sorted highest to lowest:")
print(df_idf.to_string(index=False))
print()

# Visualise IDF
fig, ax = plt.subplots(figsize=(10, 4))
colors = ['#e74c3c' if v > df_idf['IDF'].median() else '#3498db' for v in df_idf['IDF']]
ax.bar(df_idf['Term'], df_idf['IDF'], color=colors)
ax.set_title('IDF Values — Red = above median (more discriminative)')
ax.set_xlabel('Term')
ax.set_ylabel('IDF')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()


### Interpretation
- Terms with **high IDF** (red bars) are rare across the corpus — they are strong discriminators between documents. Examples: *ransomware*, *injection*, *overflow*.
- Terms with **low IDF** (blue bars) appear in almost every document — *vulnerability*, *attacker*, *remote*. These are generic CVE vocabulary that don't distinguish one vulnerability from another.
- IDF is the mechanism that makes TF-IDF a better representation than raw TF alone.


## Step 6 — TF-IDF Weighting

**TF-IDF** combines both steps:

$$\text{TF-IDF}(t, d) = tf(t, d) \times idf_t$$

A term scores high only when it is **frequent in a specific document** AND **rare across the corpus** — exactly what makes it a good descriptor of that document's topic.


In [ ]:
# Compute TF-IDF manually
tfidf_manual = tf_mat * idf_vals

df_tfidf_manual = pd.DataFrame(
    tfidf_manual[:6],
    index=[f"Doc {i} [{cve_ids[i][:12]}]" for i in range(6)],
    columns=vocab_tf
)

print("TF-IDF Matrix (manual, first 6 docs, 15 terms):")
print(df_tfidf_manual.round(4).to_string())
print()

# sklearn TF-IDF for the full retrieval pipeline
vectorizer  = TfidfVectorizer(stop_words='english')
doc_vectors = vectorizer.fit_transform(documents)

print(f"sklearn TF-IDF matrix shape : {doc_vectors.shape}")
print(f"Vector space                : R^{doc_vectors.shape[1]}")
print()

# Compare representations for one document
print("Comparison for Doc 0 — top 5 weighted terms per representation:")
terms = cv_tf.get_feature_names_out()
for label, matrix in [('Raw TF', tf_mat[0]), ('Log TF', log_tf[0]), ('TF-IDF', tfidf_manual[0])]:
    top5 = sorted(zip(terms, matrix), key=lambda x: -x[1])[:5]
    top5_str = ', '.join([f"{t}={v:.3f}" for t,v in top5 if v > 0])
    print(f"  {label:<10}: {top5_str}")


### Interpretation
- Raw TF gives equal weight to *vulnerability* (generic) and *injection* (specific) if they appear the same number of times.
- TF-IDF suppresses *vulnerability* (low IDF — appears everywhere) and amplifies *injection* (high IDF — appears rarely).
- The result is a more meaningful document vector where **topic-specific terms dominate**.
- This directly improves cosine similarity retrieval — documents about SQL injection become more similar to each other than to unrelated vulnerabilities.


## Step 7 — Cosine Similarity Retrieval

**Cosine similarity** measures the angle between two vectors in the TF-IDF space:

$$\cos(\vec{q}, \vec{d}) = \frac{\vec{q} \cdot \vec{d}}{\|\vec{q}\| \cdot \|\vec{d}\|}$$

- Score of **1.0** = identical direction (very similar)
- Score of **0.0** = orthogonal (no shared vocabulary)

We define **5 information needs** representing realistic SOC analyst queries.


In [ ]:
# --- 5 Information Needs ---
queries = [
    ("Q1: Remote Code Execution",  "remote code execution arbitrary"),
    ("Q2: Buffer Overflow",        "buffer overflow memory corruption"),
    ("Q3: SQL Injection",          "sql injection database web"),
    ("Q4: Ransomware Campaign",    "ransomware encryption file attack"),
    ("Q5: Authentication Bypass",  "authentication bypass privilege credentials"),
]

print("Information needs and top-3 retrieved documents:")
print("=" * 95)

query_results = {}  # store for later evaluation steps

for q_name, q_text in queries:
    q_vec  = vectorizer.transform([q_text])
    sims   = cosine_similarity(q_vec, doc_vectors)[0]
    ranked = np.argsort(sims)[::-1]
    query_results[q_name] = {'sims': sims, 'ranked': ranked, 'text': q_text}

    print(f"\n{q_name}: "{q_text}"")
    print(f"  {'Rank':<6} {'CVE ID':<18} {'Sim':>7}  {'Relevant?':<12} Topic")
    print(f"  {'-'*85}")
    for rank, idx in enumerate(ranked[:5], 1):
        rel = 'RELEVANT' if idx in truly_relevant else 'not relevant'
        print(f"  {rank:<6} {cve_ids[idx]:<18} {sims[idx]:>7.4f}  {rel:<12} {topics[idx][:45]}")


## Part C — Vector Space Visualisation (PCA)

In [ ]:
# PCA projection for Q1 (Remote Code Execution)
q_name = "Q1: Remote Code Execution"
q_vec  = vectorizer.transform([query_results[q_name]['text']])
sims   = query_results[q_name]['sims']
ranked = query_results[q_name]['ranked']

doc_dense   = doc_vectors.toarray()
query_dense = q_vec.toarray()
combined    = np.vstack([doc_dense, query_dense])

pca         = PCA(n_components=2)
reduced     = pca.fit_transform(combined)
doc_pts     = reduced[:len(documents)]
query_pt    = reduced[-1]

point_colors = ['#2ecc71' if i in truly_relevant else '#bdc3c7'
                for i in range(len(documents))]

fig, ax = plt.subplots(figsize=(11, 7))
ax.scatter(doc_pts[:,0], doc_pts[:,1], c=point_colors, s=35, alpha=0.7, zorder=3)

for rank, idx in enumerate(ranked[:5], 1):
    ax.scatter(doc_pts[idx,0], doc_pts[idx,1], s=100, color='#e74c3c', zorder=4)
    ax.text(doc_pts[idx,0], doc_pts[idx,1]+0.002, f'#{rank}', fontsize=7,
            ha='center', color='#c0392b', fontweight='bold')

ax.scatter(query_pt[0], query_pt[1], marker='*', s=300, color='black', zorder=5)
ax.text(query_pt[0], query_pt[1]+0.002, 'Query', fontsize=9,
        fontweight='bold', ha='center')
ax.plot([query_pt[0], doc_pts[ranked[0],0]],
        [query_pt[1], doc_pts[ranked[0],1]],
        linewidth=2.5, color='#e74c3c',
        label=f'Most similar: {cve_ids[ranked[0]]} (sim={sims[ranked[0]]:.4f})')

legend_elems = [
    mpatches.Patch(color='#2ecc71', label='Known ransomware (Relevant)'),
    mpatches.Patch(color='#bdc3c7', label='Unknown (Not Relevant)'),
    mpatches.Patch(color='#e74c3c', label='Top 5 ranked'),
]
ax.legend(handles=legend_elems, fontsize=8, loc='lower right')
ax.set_title(f'{q_name} — TF-IDF Vector Space (PCA projection)', fontsize=12)
ax.set_xlabel('Principal Component 1')
ax.set_ylabel('Principal Component 2')
plt.tight_layout()
plt.show()


## Part C — Comparing Representations: Binary vs TF-IDF

A key requirement is to compare at least **two different representations** for at least one query. We compare **Binary Incidence** against **TF-IDF** for Q1 (Remote Code Execution).


In [ ]:
# Binary retrieval
cv_bin_full = CountVectorizer(binary=True, stop_words='english')
bin_matrix  = cv_bin_full.fit_transform(documents)
q_bin       = cv_bin_full.transform([query_results['Q1: Remote Code Execution']['text']])
sims_binary = cosine_similarity(q_bin, bin_matrix)[0]
ranked_bin  = np.argsort(sims_binary)[::-1]

# TF-IDF retrieval (already computed)
sims_tfidf  = query_results['Q1: Remote Code Execution']['sims']
ranked_tfidf = query_results['Q1: Remote Code Execution']['ranked']

print("Comparison — Binary vs TF-IDF — Q1: Remote Code Execution")
print(f"{'Rank':<6} {'Binary CVE':<20} {'B-Sim':>7}  {'TF-IDF CVE':<20} {'T-Sim':>7}")
print("-" * 65)
for rank in range(10):
    b_idx = ranked_bin[rank]
    t_idx = ranked_tfidf[rank]
    b_rel = '✅' if b_idx in truly_relevant else '  '
    t_rel = '✅' if t_idx in truly_relevant else '  '
    print(f"{rank+1:<6} {b_rel}{cve_ids[b_idx]:<18} {sims_binary[b_idx]:>7.4f}  "
          f"{t_rel}{cve_ids[t_idx]:<18} {sims_tfidf[t_idx]:>7.4f}")

# Precision@5 for both
p5_bin   = sum(1 for idx in ranked_bin[:5]   if idx in truly_relevant) / 5
p5_tfidf = sum(1 for idx in ranked_tfidf[:5] if idx in truly_relevant) / 5
print(f"\nPrecision@5 — Binary: {p5_bin:.2f}  |  TF-IDF: {p5_tfidf:.2f}")
print(f"TF-IDF {'outperforms' if p5_tfidf >= p5_bin else 'underperforms vs'} Binary at top-5 for this query.")


### Interpretation
- **Binary representation** treats all present terms equally — a document mentioning *execution* once ranks the same as one mentioning it five times.
- **TF-IDF** rewards documents where the query terms are both frequent *and* rare across the corpus — it surfaces more topically focused documents.
- For short vulnerability descriptions (~27 words avg), the difference is smaller than for longer documents, but TF-IDF consistently surfaces more relevant CVEs in the top positions.


## Part D — Evaluation

We evaluate retrieval for **3 queries**: Q1 (Remote Code Execution), Q3 (SQL Injection), and Q4 (Ransomware Campaign).

### Relevance Judgments
Ground truth relevance comes from `knownRansomwareCampaignUse == 'Known'` — CISA's own threat intelligence labelling. Documents confirmed to have been used in ransomware campaigns are marked **Relevant**.

### Threshold
Documents with cosine similarity ≥ **0.10** are predicted relevant.


In [ ]:
THRESHOLD = 0.10
eval_queries = [
    "Q1: Remote Code Execution",
    "Q3: SQL Injection",
    "Q4: Ransomware Campaign"
]

y_true_global = [1 if i in truly_relevant else 0 for i in range(len(documents))]

eval_results = {}

print("=" * 80)
print(f"{'Metric':<22} {'Q1: RCE':>14} {'Q3: SQL Inj':>14} {'Q4: Ransomware':>14}")
print("=" * 80)

metric_rows = {
    'Confusion Matrix': [], 'Precision': [], 'Recall': [],
    'F1-Score': [], 'Accuracy': [], 'Kappa': [],
    'Precision@5': [], 'Precision@10': [],
    'Avg Precision (AP)': [], 'MRR': []
}

for q_name in eval_queries:
    sims   = query_results[q_name]['sims']
    ranked = query_results[q_name]['ranked']

    y_pred = [1 if sims[i] >= THRESHOLD else 0 for i in range(len(documents))]

    cm              = confusion_matrix(y_true_global, y_pred)
    tn, fp, fn, tp  = cm.ravel()
    prec  = precision_score(y_true_global, y_pred, zero_division=0)
    rec   = recall_score(y_true_global, y_pred, zero_division=0)
    f1    = f1_score(y_true_global, y_pred, zero_division=0)
    acc   = accuracy_score(y_true_global, y_pred)
    kappa = cohen_kappa_score(y_true_global, y_pred)

    p5  = sum(1 for idx in ranked[:5]  if idx in truly_relevant) / 5
    p10 = sum(1 for idx in ranked[:10] if idx in truly_relevant) / 10

    found=0; ap_sum=0.0; mrr=0.0
    for k, idx in enumerate(ranked, 1):
        if idx in truly_relevant:
            found += 1
            ap_sum += found / k
            if mrr == 0: mrr = 1.0 / k
    ap = ap_sum / total_relevant

    eval_results[q_name] = {
        'cm': cm, 'tp': tp, 'fp': fp, 'fn': fn, 'tn': tn,
        'prec': prec, 'rec': rec, 'f1': f1, 'acc': acc,
        'kappa': kappa, 'p5': p5, 'p10': p10, 'ap': ap, 'mrr': mrr,
        'y_pred': y_pred, 'sims': sims, 'ranked': ranked
    }

    metric_rows['Confusion Matrix'].append(f"TP={tp} FP={fp} FN={fn} TN={tn}")
    metric_rows['Precision'].append(f"{prec:.4f}")
    metric_rows['Recall'].append(f"{rec:.4f}")
    metric_rows['F1-Score'].append(f"{f1:.4f}")
    metric_rows['Accuracy'].append(f"{acc:.4f}")
    metric_rows['Kappa'].append(f"{kappa:.4f}")
    metric_rows['Precision@5'].append(f"{p5:.4f}")
    metric_rows['Precision@10'].append(f"{p10:.4f}")
    metric_rows['Avg Precision (AP)'].append(f"{ap:.4f}")
    metric_rows['MRR'].append(f"{mrr:.4f}")

for metric, vals in metric_rows.items():
    print(f"{metric:<22} {vals[0]:>14} {vals[1]:>14} {vals[2]:>14}")

print("=" * 80)


### Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, q_name in zip(axes, eval_queries):
    r   = eval_results[q_name]
    cm  = r['cm']
    kap = r['kappa']
    im  = ax.imshow(cm, cmap='Blues', interpolation='nearest')
    plt.colorbar(im, ax=ax)
    ax.set_xticks([0,1]); ax.set_yticks([0,1])
    ax.set_xticklabels(['Pred: Not Rel', 'Pred: Relevant'], fontsize=8)
    ax.set_yticklabels(['Act: Not Rel', 'Act: Relevant'], fontsize=8)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center',
                    fontsize=16, fontweight='bold',
                    color='white' if cm[i,j] > cm.max()/2 else 'black')
    ax.set_title(f'{q_name}\nKappa={kap:.4f}', fontsize=9)

plt.suptitle('Confusion Matrices — threshold = 0.10', fontsize=12)
plt.tight_layout()
plt.show()


### Precision@K and Recall@K

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, q_name in zip(axes, eval_queries):
    ranked = eval_results[q_name]['ranked']
    found_pk=0; found_rk=0
    pk_vals=[]; rk_vals=[]; labels=[]
    for k, idx in enumerate(ranked, 1):
        if idx in truly_relevant:
            found_pk+=1; found_rk+=1
            labels.append('Relevant')
        else:
            labels.append('Not Relevant')
        pk_vals.append(found_pk/k)
        rk_vals.append(found_rk/total_relevant)

    bar_colors = ['#2ecc71' if l=='Relevant' else '#e74c3c' for l in labels]
    ax.bar(range(1,len(documents)+1), pk_vals,
           color=bar_colors, edgecolor='none', width=1.0)
    ax.axhline(y=total_relevant/len(documents), color='black',
               linestyle='dashed', alpha=0.5, linewidth=1)
    ax.set_title(f'{q_name}\nP@5={eval_results[q_name]["p5"]:.2f}  P@10={eval_results[q_name]["p10"]:.2f}',
                 fontsize=9)
    ax.set_xlabel('Rank K'); ax.set_ylabel('Precision@K')
    ax.set_ylim(0, 1.1)

plt.suptitle('Precision@K — Green=Relevant at rank, Red=Not Relevant, Dashed=Random baseline',
             fontsize=10)
plt.tight_layout()
plt.show()


### Average Precision (AP) and Precision-Recall Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, q_name in zip(axes, eval_queries):
    ranked = eval_results[q_name]['ranked']
    ap     = eval_results[q_name]['ap']
    found_pk=0; found_rk=0
    pk_vals=[]; rk_vals=[]
    ap_pk=[]; ap_rk=[]
    for k, idx in enumerate(ranked, 1):
        if idx in truly_relevant:
            found_pk+=1; found_rk+=1
            ap_pk.append(found_pk/k)
            ap_rk.append(found_rk/total_relevant)
        pk_vals.append(found_pk/k)
        rk_vals.append(found_rk/total_relevant)

    ax.plot(rk_vals, pk_vals, color='#2980b9', linewidth=1.5, alpha=0.5)
    ax.scatter(ap_rk, ap_pk, color='#e74c3c', s=50, zorder=5,
               label='AP sample points')
    ax.fill_between(ap_rk, ap_pk, alpha=0.2, color='#e74c3c')
    ax.axhline(y=ap, color='black', linestyle='dashed', alpha=0.7,
               label=f'AP = {ap:.4f}')
    ax.set_title(f'{q_name}\nAP = {ap:.4f}', fontsize=9)
    ax.set_xlabel('Recall@K'); ax.set_ylabel('Precision@K')
    ax.set_xlim(0, 1.05); ax.set_ylim(0, 1.1)
    ax.legend(fontsize=7)

plt.suptitle('Precision-Recall Curves with Average Precision', fontsize=11)
plt.tight_layout()
plt.show()


### Mean Reciprocal Rank (MRR)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3))

for ax, q_name in zip(axes, eval_queries):
    ranked = eval_results[q_name]['ranked']
    mrr    = eval_results[q_name]['mrr']
    first_rel_rank = int(round(1.0/mrr)) if mrr > 0 else len(documents)

    bar_colors = ['#2ecc71' if ranked[i] in truly_relevant else '#bdc3c7'
                  for i in range(min(30, len(documents)))]
    if first_rel_rank <= 30:
        bar_colors[first_rel_rank-1] = '#e74c3c'

    ax.bar(range(1, 31), [1]*30, color=bar_colors, edgecolor='white', linewidth=0.3)
    if first_rel_rank <= 30:
        ax.text(first_rel_rank, 1.05,
                f'Rank {first_rel_rank}\nMRR={mrr:.2f}',
                ha='center', fontsize=8, color='#c0392b', fontweight='bold')
    ax.set_title(f'{q_name}\nMRR = {mrr:.4f}', fontsize=9)
    ax.set_xlabel('Rank K'); ax.set_yticks([])
    ax.set_xlim(0, 31)

plt.suptitle('MRR — Red = first relevant doc, Green = other relevant docs (first 30 ranks)',
             fontsize=10)
plt.tight_layout()
plt.show()

print("MRR Summary:")
for q_name in eval_queries:
    mrr = eval_results[q_name]['mrr']
    first = int(round(1.0/mrr)) if mrr > 0 else 'not found'
    print(f"  {q_name:<30} MRR={mrr:.4f}  First relevant doc at rank {first}")


### Kappa (Cohen's Kappa)

In a team setting, two judges independently label the same documents for one query. Kappa measures agreement **adjusted for chance**.

Here we simulate two judges by applying two different thresholds to the same similarity scores.


In [ ]:
# Simulate two judges with slightly different thresholds on Q1
q_name  = "Q1: Remote Code Execution"
sims    = eval_results[q_name]['sims']

judge1 = [1 if sims[i] >= 0.10 else 0 for i in range(len(documents))]
judge2 = [1 if sims[i] >= 0.15 else 0 for i in range(len(documents))]

kappa_judges = cohen_kappa_score(judge1, judge2)
kappa_vs_gt  = eval_results[q_name]['kappa']

print(f"Query: {q_name}")
print(f"Judge 1 (threshold=0.10) retrieved : {sum(judge1)} documents")
print(f"Judge 2 (threshold=0.15) retrieved : {sum(judge2)} documents")
print(f"Kappa between Judge 1 and Judge 2  : {kappa_judges:.4f}")
print(f"Kappa vs ground truth (Judge 1)    : {kappa_vs_gt:.4f}")
print()

kappa_scale = [
    (0.0,  "Slight agreement"),
    (0.2,  "Fair agreement"),
    (0.4,  "Moderate agreement"),
    (0.6,  "Substantial agreement"),
    (0.8,  "Almost perfect agreement")
]
def interpret_kappa(k):
    for threshold, label in reversed(kappa_scale):
        if k >= threshold: return label
    return "Worse than random"

print(f"Interpretation (Judge1 vs Judge2): {interpret_kappa(kappa_judges)}")
print(f"Interpretation (vs ground truth): {interpret_kappa(kappa_vs_gt)}")


## Summary — All Metrics Across 3 Queries

In [ ]:
print("=" * 90)
print("EVALUATION SUMMARY — CISA KEV Corpus")
print(f"Dataset : CISA Known Exploited Vulnerabilities Catalog")
print(f"Source  : https://www.cisa.gov/sites/default/files/csv/known_exploited_vulnerabilities.csv")
print(f"Docs    : {len(documents)} sampled | Relevant: {total_relevant} | Threshold: {THRESHOLD}")
print("=" * 90)
print(f"{'Metric':<22} {'Q1: RCE':>16} {'Q3: SQL Inj':>16} {'Q4: Ransomware':>16}")
print("-" * 74)

for metric_name, key in [
    ('Precision',        'prec'),
    ('Recall',           'rec'),
    ('F1-Score',         'f1'),
    ('Accuracy',         'acc'),
    ('Kappa',            'kappa'),
    ('Precision@5',      'p5'),
    ('Precision@10',     'p10'),
    ('Avg Precision AP', 'ap'),
    ('MRR',              'mrr'),
]:
    vals = [f"{eval_results[q][key]:.4f}" for q in eval_queries]
    print(f"{metric_name:<22} {vals[0]:>16} {vals[1]:>16} {vals[2]:>16}")

print("=" * 90)


## Reflection

### 1. Which representation worked best and why?
**TF-IDF outperformed Binary** representation in our experiments. Binary treats all present terms equally, so a document mentioning *execution* once ranks the same as one focused entirely on execution. TF-IDF amplifies terms that are frequent in a specific document but rare across the corpus — exactly the terms that define what a vulnerability document is *about*.

### 2. Did TF-IDF improve over raw term counts?
Yes. Raw TF gave excessive weight to generic CVE vocabulary like *vulnerability*, *attacker*, and *contains* — words that appear in nearly every document and carry no discriminative power. TF-IDF suppresses these through IDF weighting and surfaces domain-specific terms like *ransomware*, *injection*, and *overflow* that better characterise individual vulnerability categories.

### 3. What kinds of false positives did you observe?
The most common false positives were vulnerabilities from unrelated categories that happened to share generic terms with the query. For example, a query for *remote code execution* would sometimes retrieve authentication bypass vulnerabilities because both descriptions contain *remote* and *attacker*. The problem is that short descriptions (avg 27 words) share a limited vocabulary, making it harder for TF-IDF to separate them.

### 4. What kinds of relevant documents were missed (false negatives)?
Several Known ransomware CVEs used unusual or vendor-specific terminology not present in the query. For example, a ransomware-used vulnerability described primarily in terms of *privilege escalation* or *arbitrary file write* would not surface for a query containing *ransomware encryption file attack* — the information need is the same but the vocabulary does not overlap.

### 5. How did the evaluation metrics help understand system quality?
The metrics together told a more complete story than any single number could:
- **Precision@5** revealed that the top of the ranking was often good even when overall Precision was moderate.
- **AP** showed that Q4 (Ransomware) had the best overall ranking quality — relevant docs were concentrated near the top.
- **MRR** confirmed that the system reliably finds *at least one* relevant document quickly for all three queries.
- **Kappa** exposed the class imbalance problem — high accuracy was partly an artefact of the 80/20 not-relevant/relevant split.

### 6. How would you improve the system?
1. **Dense embeddings** — replace TF-IDF with sentence-transformers trained on security text. Short vulnerability descriptions benefit enormously from semantic representations that understand *code execution* and *arbitrary command injection* as related concepts.
2. **Query expansion** — expand the query with synonyms (e.g. *RCE* → *remote code execution*, *arbitrary command execution*) to reduce vocabulary mismatch.
3. **Larger sample** — use all 1,551 CVEs rather than 200. More documents means richer IDF values and more discriminative vectors.
4. **Domain-specific stop words** — add *vulnerability*, *attacker*, *allows*, *contains* to the stop list since they appear in virtually every CVE description.
5. **Relevance feedback** — use initially retrieved relevant documents to refine the query vector (Rocchio algorithm).


---
## Team
- Emmanuel Ihejiamaizu (Chooks)
- Liggia Elena Taboada Cruz  
- Chao-Chung Liu (Thomas)

**Group 6 | PROG 8245 — Machine Learning Programming | Conestoga College**
